# Deepeval metric experiments

Every metric has a separate explanation and test cell. Shared fixtures provide 10 RAG and 10 agent examples. These cells prepare labeled inputs; actual judge scores require the selected framework evaluator and model.

In [ ]:
import json
from pathlib import Path

experiment = {'framework': 'DeepEval', 'rag_metrics': ['faithfulness', 'answer_relevancy', 'contextual_relevancy', 'contextual_precision', 'contextual_recall', 'hallucination', 'GEval_correctness'], 'agent_metrics': ['tool_correctness', 'argument_correctness', 'task_completion', 'step_efficiency', 'role_adherence'], 'rag_case_fields': ['input', 'actual_output', 'expected_output', 'retrieval_context'], 'agent_case_fields': ['input', 'actual_output', 'expected_output', 'tools_called', 'expected_tools'], 'status': 'plan_only: wire test cases and metrics into CI'}

def load_shared_cases(filename):
    for folder in (Path.cwd(), *Path.cwd().parents):
        for candidate in (folder / 'datasets' / filename, folder / 'opensource' / 'datasets' / filename):
            if candidate.is_file():
                return json.loads(candidate.read_text(encoding='utf-8'))
    raise FileNotFoundError(f'Cannot find shared dataset: {filename}')

rag_cases = load_shared_cases('rag_cases.json')
agent_cases = load_shared_cases('agent_cases.json')
assert len(rag_cases) >= 10 and len(agent_cases) >= 10
assert all({'question','contexts','response','reference','expected_context_relevance'} <= case.keys() for case in rag_cases)
assert all({'task','tool_trace','expected_tools','expected_outcome','final_response'} <= case.keys() for case in agent_cases)
print(f'Loaded {len(rag_cases)} RAG and {len(agent_cases)} agentic AI examples.')


## RAG metrics

The metric checks below are isolated so you can inspect and run them individually.

### Faithfulness

**What it measures:** Checks whether answer claims are supported by retrieved passages.

**Why/when to use it:** Use it to expose unsupported details; a judge compares response and contexts.

**Example fields:** question, contexts, response.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'faithfulness'
metric_fields = ['question', 'contexts', 'response']
metric_label = 'expected_grounded'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Answer Relevancy

**What it measures:** Checks whether the response directly addresses the user question.

**Why/when to use it:** Use it to catch off-topic or evasive answers; relevance is distinct from correctness.

**Example fields:** question, response.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'answer_relevancy'
metric_fields = ['question', 'response']
metric_label = 'expected_relevant'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Contextual Relevancy

**What it measures:** Scores how much retrieved context is relevant to the input.

**Why/when to use it:** Use it to diagnose noisy retrieval independently of answer generation.

**Example fields:** question, contexts.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'contextual_relevancy'
metric_fields = ['question', 'contexts']
metric_label = 'expected_context_relevance'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Contextual Precision

**What it measures:** Measures whether relevant retrieved passages rank ahead of irrelevant ones.

**Why/when to use it:** Use it to evaluate retrieval ordering and reranking.

**Example fields:** question, contexts, expected_context_relevance.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'contextual_precision'
metric_fields = ['question', 'contexts', 'expected_context_relevance']
metric_label = 'expected_context_relevance'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Contextual Recall

**What it measures:** Measures whether retrieved context covers facts required by the reference.

**Why/when to use it:** Use it to diagnose retrieval omissions; requires reference or evidence annotations.

**Example fields:** question, contexts, reference.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'contextual_recall'
metric_fields = ['question', 'contexts', 'reference']
metric_label = None
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Hallucination

**What it measures:** Detects claims absent from or contradicted by the available evidence.

**Why/when to use it:** Use it to find fabricated details; review individual claims rather than only aggregate scores.

**Example fields:** response, contexts.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'hallucination'
metric_fields = ['response', 'contexts']
metric_label = 'expected_grounded'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Geval Correctness

**What it measures:** Applies a natural-language grading rubric to compare answer and expected answer.

**Why/when to use it:** Use it for nuanced criteria that exact string matching misses; calibrate rubric judgments with humans.

**Example fields:** question, response, reference.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'GEval_correctness'
metric_fields = ['question', 'response', 'reference']
metric_label = None
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


## Agentic AI metrics

The metric checks below are isolated so you can inspect and run them individually.

### Tool Correctness

**What it measures:** Checks whether tools used by the agent were appropriate for the task.

**Why/when to use it:** Use it to catch wrong or missing actions; define how valid alternatives are treated.

**Example fields:** task, tool_trace, expected_tools.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'tool_correctness'
metric_fields = ['task', 'tool_trace', 'expected_tools']
metric_label = 'expected_tool_correctness'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Argument Correctness

**What it measures:** Checks whether tool-call arguments correctly reflect the user request.

**Why/when to use it:** Use it to catch parameter errors before they cause wrong side effects.

**Example fields:** task, tool_trace.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'argument_correctness'
metric_fields = ['task', 'tool_trace']
metric_label = 'expected_argument_correctness'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Task Completion

**What it measures:** Checks whether the final result satisfies task requirements.

**Why/when to use it:** Use it to distinguish fluent output from an actually completed task.

**Example fields:** task, final_response, expected_outcome.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'task_completion'
metric_fields = ['task', 'final_response', 'expected_outcome']
metric_label = 'expected_task_success'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Step Efficiency

**What it measures:** Checks whether the agent avoids needless steps, retries, and duplicate calls.

**Why/when to use it:** Use it to control latency/cost without rewarding skipped safety checks.

**Example fields:** task, tool_trace, max_steps.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'step_efficiency'
metric_fields = ['task', 'tool_trace', 'max_steps']
metric_label = None
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Role Adherence

**What it measures:** Checks whether the agent stays within its assigned role and constraints.

**Why/when to use it:** Use role-specific rubrics for behavior that may be correct but outside the agent remit.

**Example fields:** task, final_response.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'role_adherence'
metric_fields = ['task', 'final_response']
metric_label = None
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})
